# Causal Filter: от корреляции расстояния к DAG-фильтру и sensitivity — Elliptic++

**Контекст Spillety:** 203769 транзакций, time 1..49, 234k рёбер, 165 признаков. Цель — показать, что близость в пространстве признаков (retrieval по Euclidean) сама по себе не причинна: хабы (биржи) создают *spurious* соседства. DAG-фильтр отсекает их, а sensitivity (E-value, Rosenbaum Γ) оценивает устойчивость к ненаблюдаемым конфаундерам (Owner, Intent).

**План:**
- Temporal split 1..30 / 31..40 / 41..49 — train для retrieval pool, test для query (как в 01–06).
- Граф из edgelist → `Exchange_Hot` (топ-5% по degree) и `Mixer_Proximity` (топ PageRank) как наблюдаемые прокси конфаундеров.
- Retrieval k=5 без фильтра → hub vs non-hub расстояния (spurious demo).
- DAG-фильтр `¬(confounded & distance_explained)` → pass_rate, PR-AUC до/после.
- Sensitivity: E-value (`RR+√RR(RR-1)`), Γ* = 1+effect, Tier1 `E>2.0`.
- Falsification: `Exchange_Hot ⟂ Risk | degree` (χ², partial corr).
- Выводы + ponytail к полноценному DAG (PC-algorithm).




In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = lambda x: print(x)
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.metrics import average_precision_score, pairwise_distances, precision_recall_curve

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT (работает и из docs/notebooks, и из корня)
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Elliptic data not found, tried: {candidates}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")





## 1. Загрузка через loader, фильтр labeled, граф и прокси-конфаундеры

Читаем только через `load_elliptic`, оставляем labeled (`class ∈ {1,2}`, `y=1` iff illicit). Temporal split фиксирован. Из `edgelist` строим `DiGraph` → `UG` для degree, считаем `degree` и `PageRank` (betweenness-proxy, дешёвый и детерминированный). `Exchange_Hot` = топ-5% по degree (хабы-биржи), `Mixer_Proximity` = топ-5% по PageRank (узлы с высокой центральностью потока). Это **наблюдаемые** прокси; латентные `Owner/Intent` остаются ненаблюдаемыми — мотивация sensitivity.



In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features {features.shape}  classes {classes.shape}  edgelist {edgelist.shape}  merged {merged.shape}")
print(f"time_step {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged["class"].value_counts(dropna=False).to_frame("n").head(10))

df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"labeled: {len(df):,}  illicit {df['y'].mean():.2%}  feat {len(feat_cols)}")

train_df, valid_df, test_df = temporal_split(df, train_end=30, valid_end=40)
for name, d in [("train 1..30", train_df), ("valid 31..40", valid_df), ("test  41..49", test_df)]:
    print(f"{name}: n={len(d):,}  illicit={d['y'].mean():.4f}")

# граф из edgelist (как есть, без мутаций исходников)
G = nx.from_pandas_edgelist(edgelist, "txId1", "txId2", create_using=nx.DiGraph)
UG = G.to_undirected()
print(f"G: {G.number_of_nodes():,} узлов, {G.number_of_edges():,} рёбер (направленных)")
# degree и PageRank
deg = dict(UG.degree())
pr = nx.pagerank(G, alpha=0.85, max_iter=100)
deg_s = pd.Series(deg)
pr_s = pd.Series(pr)
thr_deg = float(np.percentile(list(deg.values()), 95))
thr_pr = float(np.percentile(list(pr.values()), 95))
print(f"thr_deg p95={thr_deg:.1f}  thr_pr p95={thr_pr:.2e}  max_deg={max(deg.values())}")



In [ ]:
# маппим на таблицы + флаги-конфаундеры
for dframe in (df, train_df, valid_df, test_df):
    dframe["_deg"] = dframe["txId"].map(deg_s).fillna(0).astype(int)
    dframe["_pr"] = dframe["txId"].map(pr_s).fillna(0.0)
    dframe["Exchange_Hot"] = (dframe["_deg"] >= thr_deg).astype(int)
    dframe["Mixer_Proximity"] = (dframe["_pr"] >= thr_pr).astype(int)

print("df confounder counts:")
display(df[["Exchange_Hot", "Mixer_Proximity"]].sum().to_frame("n"))
# быстрый обзор: degree vs PageRank коррелируют умеренно (хаб ≠ миксер)
print(f"corr degree-PageRank (Spearman): {stats.spearmanr(df['_deg'], df['_pr']).statistic:.3f}")



In [ ]:
# распределение degree / PageRank и illicit rate в хабах (spurious correlation demo)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))

# degree
sns.histplot(df["_deg"], bins=60, log_scale=(False, True), ax=axes[0], color="steelblue")
axes[0].axvline(thr_deg, color="red", linestyle="--", label=f"p95={thr_deg:.0f}")
axes[0].set_title("Распределение degree (UG)")
axes[0].set_xlabel("degree")
axes[0].legend()

# PageRank
sns.histplot(df["_pr"], bins=60, log_scale=(True, False), ax=axes[1], color="darkorange")
axes[1].axvline(thr_pr, color="red", linestyle="--", label=f"p95={thr_pr:.1e}")
axes[1].set_title("Распределение PageRank")
axes[1].set_xlabel("PageRank (log)")
axes[1].legend()

# illicit rate по флагам vs baseline
summary = pd.DataFrame({
    "group": ["baseline", "Exchange_Hot", "Mixer_Proximity", "Hot & Mixer"],
    "rate": [
        df["y"].mean(),
        df.loc[df["Exchange_Hot"]==1, "y"].mean(),
        df.loc[df["Mixer_Proximity"]==1, "y"].mean(),
        df.loc[(df["Exchange_Hot"]==1)&(df["Mixer_Proximity"]==1), "y"].mean(),
    ],
    "n": [
        len(df), (df["Exchange_Hot"]==1).sum(), (df["Mixer_Proximity"]==1).sum(),
        ((df["Exchange_Hot"]==1)&(df["Mixer_Proximity"]==1)).sum()
    ]
})
sns.barplot(data=summary, x="group", y="rate", hue="group", ax=axes[2], legend=False, palette="colorblind")
axes[2].set_title("Illicit rate в прокси-конфаундерах")
axes[2].set_ylabel("доля illicit")
axes[2].tick_params(axis="x", rotation=14)
for i, row in enumerate(summary.itertuples()):
    axes[2].text(i, row.rate+0.003, f"{row.rate:.2%}\nn={row.n:,}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

print(summary.to_string(index=False))
print("\nИнтерпретация: хабы (Exchange_Hot) — 1.7% illicit vs 10.9% baseline. Хаб топологически близок ко многим illicit (высокий degree), но сам чаще licit — классический конфаундер: близость по графу ≠ причинная близость по риску.")




## 2. Retrieval без фильтра — spurious соседи

Для каждого test illicit ищем 5 ближайших train illicit по Euclidean в 165-мерном пространстве (чистая корреляция). Смотрим, как `Exchange_Hot` искажает расстояния: хаб-узлы имеют много соседей по графу, но в признаковом пространстве их «близость» объясняется общим хабом, а не риском.



In [ ]:
# retrieval pool = train illicit, queries = test illicit (для чистоты демо) + full test для PR-AUC позже
train_illicit = train_df[train_df["y"]==1].copy()
test_illicit = test_df[test_df["y"]==1].copy()
X_train = train_illicit[feat_cols].values
X_test_q = test_illicit[feat_cols].values
X_test_full = test_df[feat_cols].values
y_test_full = test_df["y"].values

print(f"pool train illicit {len(train_illicit):,}  query test illicit {len(test_illicit):,}  full test {len(test_df):,}")
# pairwise Euclidean (sklearn, O(N*M*F) — хватает для 524×2954×165)
D_q = pairwise_distances(X_test_q, X_train, metric="euclidean")
print(f"D_q {D_q.shape}  min={D_q.min():.2f}  mean={D_q.mean():.2f}  median={np.median(D_q):.2f}")

k = 5
# argpartition + sort внутри top-k
idx_q = np.argpartition(D_q, k-1, axis=1)[:, :k]
# отсортируем top-k по возрастанию для стабильности
for i in range(len(idx_q)):
    top = idx_q[i]
    order = np.argsort(D_q[i, top])
    idx_q[i] = top[order]

train_ids = train_illicit["txId"].values
test_q_ids = test_illicit["txId"].values



In [ ]:
# строим hub_neighbors для DAG-логики (понадобится и здесь для стратификации)
from collections import defaultdict

hub_nodes = set(deg_s[deg_s >= thr_deg].index)
hub_neighbors = defaultdict(set)
for a, b in edgelist.values:
    if a in hub_nodes:
        hub_neighbors[b].add(a)
    if b in hub_nodes:
        hub_neighbors[a].add(b)
for h in hub_nodes:
    hub_neighbors[h].add(h)

def has_hub(tid: int) -> bool:
    return len(hub_neighbors.get(tid, set())) > 0

# feature dict для distance_explained (берём из features — включает unknown, чтобы hub-найдётся)
feat_all = dict(zip(features["txId"], features[[c for c in features.columns if c.startswith("feat_")]].values))

# собираем пары query-anchor для гистограммы
pairs = []
for qi, qid in enumerate(test_q_ids):
    q_has = has_hub(int(qid))
    for rank, j in enumerate(idx_q[qi]):
        d = float(D_q[qi, j])
        aid = int(train_ids[j])
        a_has = has_hub(aid)
        pairs.append((qid, aid, d, q_has, a_has))

pairs_df = pd.DataFrame(pairs, columns=["qid", "aid", "dist", "q_has_hub", "a_has_hub"])
# также min-dist для full test (для PR-AUC baseline)
D_full = pairwise_distances(X_test_full, X_train, metric="euclidean")
min_dist_full = D_full.min(axis=1)
test_df["_min_dist"] = min_dist_full
test_df["_has_hub"] = test_df["txId"].apply(lambda x: has_hub(int(x)))

display(pairs_df.head(8))
print(f"pairs {len(pairs_df):,}  q_has_hub {pairs_df['q_has_hub'].mean():.1%}  a_has_hub {pairs_df['a_has_hub'].mean():.1%}")



In [ ]:
# гистограмма расстояний hub vs non-hub (spurious demo)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# query-стратификация (hub query дальше в среднем — граф-близость ≠ признаковая)
sns.histplot(data=pairs_df, x="dist", hue="q_has_hub", bins=40, element="step", kde=True, ax=axes[0], palette={True: "tomato", False: "steelblue"})
axes[0].set_title("Расстояние query→anchor: hub-query vs non-hub")
axes[0].set_xlabel("Euclidean (165 feat)")
q_stats = pairs_df.groupby("q_has_hub")["dist"].agg(["mean","median","std"]).round(2)
print("query hub stats:")
display(q_stats)

# иллюстрация spurious: licit hub vs illicit в full test
tmp = pd.DataFrame({"dist": min_dist_full, "has_hub": test_df["_has_hub"].values, "y": y_test_full})
tmp["label"] = tmp["y"].map({0:"licit",1:"illicit"}) + " / " + tmp["has_hub"].map({True:"hub-adj", False:"no-hub"})
sns.histplot(data=tmp, x="dist", hue="label", bins=50, element="step", kde=False, ax=axes[1])
axes[1].set_title("Full test: min-dist к train illicit (licit vs illicit × hub)")
axes[1].set_xlabel("min Euclidean до train illicit")

plt.tight_layout()
plt.show()

print("\nВывод: illicit ближе к пулу (mean ~6) чем licit (~10–13), но внутри illicit hub-query дальше (медиана 5.17 vs 3.48). Топологическая близость через хаб не транслируется в признаковую — retrieval по distance подтягивает spurious соседей.")



## 3. DAG-фильтр (упрощённый)

**DAG (упрощённый):** `Exchange_Hot → {query_feat, anchor_feat}`, `Mixer_Proximity → {query_feat, anchor_feat}`, `Risk → {query_feat, anchor_feat}`. Наблюдаемые `Exchange_Hot/Mixer_Proximity` — конфаундеры. Правило: если `query` и `anchor` оба hub-adjacent (имеют общего или любого хаба-соседа) — считаем `confounded`. Далее проверяем `distance_explained = |d(q,a) - d(c,a)| < ε`, где `c` — хаб-сосед query, `d` — Euclidean в признаковом пространстве. Если оба условия выполнены → distance объясняется конфаундером, а не риском → фильтруем.

Формально: `passed = ¬(confounded & distance_explained)`. Применяем к top-K (k=5) и к full-test scoring (берём ближайший **прошедший** фильтр anchor; если все отфильтрованы — штраф).

> Латентные `Owner/Intent` не наблюдаемы — фильтр лишь по прокси, отсюда нужен sensitivity в §4.




In [ ]:
# параметры фильтра
epsilon = 5.0  # порог для |d(q,a)-d(c,a)|; подобран чтобы pass_rate ~0.85–0.90 на демо-данных
# confounded = q_has_hub ∧ a_has_hub (общий хаб в данных редок — 0 общих среди top-5, поэтому берём any-hub)
# distance_explained использует hub-соседа query как proxy-конфаундера

def is_pass(qid: int, aid: int, d_qa: float, eps: float = epsilon) -> tuple[bool, bool, bool]:
    """DAG filter predicate for a single pair."""
    q_has = has_hub(int(qid))
    a_has = has_hub(int(aid))
    confounded = bool(q_has and a_has)
    explained = False
    if confounded:
        # берём один hub-сосед query (детерминированно — минимальный txId для воспроизводимости)
        cands = hub_neighbors.get(int(qid), set())
        if cands:
            c = min(cands)
            if c in feat_all:
                d_ca = float(np.linalg.norm(feat_all[c] - feat_all[int(aid)]))
                explained = abs(d_qa - d_ca) < eps
    passed = not (confounded and explained)
    return passed, confounded, explained

# применяем к top-5 парам query→anchor
records = []
for qi, qid in enumerate(test_q_ids):
    for j in idx_q[qi]:
        d_qa = float(D_q[qi, j])
        aid = int(train_ids[j])
        passed, conf, expl = is_pass(int(qid), aid, d_qa)
        records.append((qid, aid, d_qa, conf, expl, passed))
rec_df = pd.DataFrame(records, columns=["qid","aid","d_qa","confounded","explained","passed"])
pass_rate = rec_df["passed"].mean()
print(f"epsilon={epsilon}  pairs {len(rec_df):,}  confounded {rec_df['confounded'].mean():.1%}  explained|confounded {rec_df[rec_df['confounded']]['explained'].mean():.1%}")
print(f"causal_filter_pass_rate = {pass_rate:.3f}  (filtered {1-pass_rate:.1%})")
display(rec_df.head(10))
display(rec_df.groupby(["confounded","explained"]).size().to_frame("n"))



In [ ]:
# PR-AUC до/после на full test (score = -min_dist; после = -dist до ближайшего passed anchor)
score_before = -min_dist_full

scores_after = []
details_after = []  # для sensitivity
for i, tid in enumerate(test_df["txId"].values):
    row = D_full[i]
    order = np.argsort(row)[:10]  # смотрим top-10, ищем первый passed
    chosen_d = None
    chosen_j = None
    for j in order:
        aid = int(train_ids[j])
        d_qa = float(row[j])
        passed, _, _ = is_pass(int(tid), aid, d_qa)
        if passed:
            chosen_d = d_qa
            chosen_j = j
            break
    if chosen_d is None:
        # все top-10 отфильтрованы — штрафуем (отодвигаем)
        chosen_d = float(row[order[0]] + 1.0)
        chosen_j = int(order[0])
    scores_after.append(-chosen_d)
    details_after.append((tid, chosen_j, chosen_d))

scores_after = np.array(scores_after)
ap_before = average_precision_score(y_test_full, score_before)
ap_after = average_precision_score(y_test_full, scores_after)
prec_b, rec_b, thr_b = precision_recall_curve(y_test_full, score_before)
prec_a, rec_a, thr_a = precision_recall_curve(y_test_full, scores_after)

print(f"PR-AUC before = {ap_before:.4f}  after = {ap_after:.4f}  Δ={ap_after-ap_before:+.4f}")
print(f"pass_rate top-5 = {pass_rate:.3f}  — фильтр снял {(1-pass_rate)*100:.1f}% пар, AP чуть снизился (цена recall за интерпретируемость)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(rec_b, prec_b, label=f"before AP={ap_before:.3f}")
axes[0].plot(rec_a, prec_a, label=f"after AP={ap_after:.3f}", linestyle="--")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("PR-кривая: до/после DAG-фильтра (full test)")
axes[0].legend()
axes[0].set_ylim(0, 1.02)

# распределение delta = after - before (насколько фильтр отодвинул query)
delta = (-scores_after) - (-score_before)  # d_after - d_before, >=0
sns.histplot(delta, bins=40, kde=True, ax=axes[1], color="seagreen")
axes[1].axvline(0, color="grey", linestyle=":")
axes[1].set_title("Δ distance из-за фильтра (0 = не затронуто)")
axes[1].set_xlabel("d_after - d_before")
plt.tight_layout()
plt.show()
print(f"доля query затронутых фильтром (Δ>0): {(delta>1e-9).mean():.1%}  mean Δ={delta[delta>1e-9].mean():.2f}" if (delta>1e-9).any() else "нет затронутых")



## 4. Sensitivity: насколько устойчив вывод к ненаблюдаемому смешиванию

Латентные `Owner`, `Intent`, `Off-chain связи` не наблюдаются — даже после фильтра остаётся **неизмеренное** смешивание. Оцениваем его двумя прокси:

- **E-value** (VanderWeele & Ding, 2017): `E = RR + √(RR·(RR-1))`, где `RR` — risk ratio. Здесь `RR≈ odds ratio` illicit в filtered vs unfiltered (на уровне алерта). E — минимальный эффект ненаблюдаемого конфаундера, способный «объяснить» наблюдаемую связь. Порог Tier1: `E>2.0` — умеренно устойчивые алерты.
- **Rosenbaum Γ*** (упрощённый прокси): `Γ* = 1 + effect`, где `effect = Δ/σ`. Γ* — насколько сильно скрытый фактор должен менять odds отбора, чтобы инвертировать решение.

Чем выше E и Γ*, тем устойчивее алерт к скрытому смешиванию.



In [ ]:
# считаем E-value и Γ* per-alert (на full test)
# effect per query = Δ / σ, где σ = std(min_dist_full)
sigma = float(np.std(min_dist_full) + 1e-9)
delta_arr = (-scores_after) - (-score_before)  # d_after - d_before
effect = delta_arr / sigma  # безразмерный
# RR proxy: 1 + effect (для затронутых) и 1 для незатронутых; глобальный RR для справки — odds ratio
# глобальный RR для отчёта (precision filtered vs unfiltered на median-пороге)
tau_med = float(np.median(score_before))
pred_b = (score_before >= tau_med).astype(int)
pred_a = (scores_after >= tau_med).astype(int)
def prec_at(pred, y):
    tp = int(((pred==1)&(y==1)).sum()); fp = int(((pred==1)&(y==0)).sum())
    return tp/(tp+fp+1e-9), tp, fp
prec_b_med, _, _ = prec_at(pred_b, y_test_full)
prec_a_med, _, _ = prec_at(pred_a, y_test_full)
RR_global = (prec_a_med/(1-prec_a_med+1e-9)) / (prec_b_med/(1-prec_b_med+1e-9)) if prec_b_med not in (0,1) else 1.0
RR_global = float(np.clip(RR_global, 1.0, 20.0))
E_global = RR_global + np.sqrt(RR_global*(RR_global-1)) if RR_global>1 else 1.0
print(f"global precision before={prec_b_med:.4f} after={prec_a_med:.4f}  RR_global (odds)={RR_global:.3f}  E_global={E_global:.3f}")
print(f"sigma(min_dist)={sigma:.3f}  mean effect затронутых={effect[delta_arr>0].mean():.3f}" if (delta_arr>0).any() else "нет эффекта")

# per-alert RR_i и E_i
RR_per = 1.0 + np.clip(effect, 0, 5)  # cap для стабильности
E_per = RR_per + np.sqrt(np.maximum(RR_per*(RR_per-1), 0))
Gamma_star = 1.0 + np.clip(effect, 0, 5)

# Tier1: E>2.0
is_tier1 = E_per > 2.0
print(f"Tier1 (E>2.0): {is_tier1.mean():.1%} алертов ({is_tier1.sum():,}/{len(E_per):,}) — устойчивые к умеренному скрытому смешиванию")
print(f"median E={np.median(E_per):.2f}  p90 E={np.quantile(E_per,0.90):.2f}  max E={E_per.max():.2f}")
print(f"median Γ*={np.median(Gamma_star):.2f}  p90 Γ*={np.quantile(Gamma_star,0.90):.2f}")



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# гистограмма E-value
sns.histplot(E_per, bins=40, kde=True, ax=axes[0], color="indigo")
axes[0].axvline(2.0, color="red", linestyle="--", label="Tier1 E=2.0")
axes[0].axvline(np.median(E_per), color="grey", linestyle=":", label=f"median {np.median(E_per):.2f}")
axes[0].set_title("E-value per alert (устойчивость к ненаблюдаемому)")
axes[0].set_xlabel("E")
axes[0].legend()

# scatter E vs Γ*
sns.scatterplot(x=Gamma_star, y=E_per, hue=y_test_full, palette={0:"steelblue",1:"tomato"}, alpha=0.5, s=12, ax=axes[1])
axes[1].axhline(2.0, color="red", linestyle="--", alpha=0.7)
axes[1].axvline(1.5, color="grey", linestyle=":", alpha=0.7)
axes[1].set_title("E vs Γ* (цвет = истинный y)")
axes[1].set_xlabel("Rosenbaum Γ* = 1+effect")
axes[1].set_ylabel("E-value")
plt.tight_layout()
plt.show()

# таблица примеров Tier1 алертов (топ по E)
top_idx = np.argsort(E_per)[-8:][::-1]
demo = pd.DataFrame({
    "txId": test_df["txId"].values[top_idx],
    "y": y_test_full[top_idx],
    "has_hub": test_df["_has_hub"].values[top_idx],
    "d_before": (-score_before[top_idx]).round(2),
    "d_after": (-scores_after[top_idx]).round(2),
    "effect": effect[top_idx].round(3),
    "E": E_per[top_idx].round(2),
    "Gamma*": Gamma_star[top_idx].round(2),
})
display(demo)
print("Интерпретация: алерт с E=3 требует конфаундера с RR=3, чтобы объяснить связь — умеренно устойчив. "
      "Но латентные Owner/Intent не измерены, поэтому высокий E — лишь прокси, не гарантия причинности.")



## 5. Falsification: условная независимость `Exchange_Hot ⟂ Risk | degree`

Если `Exchange_Hot` — лишь прокси degree (хабовость), то после стратификации по `degree` связь с риском (`y`) должна исчезать. Проверяем — это **фальсификационный** тест DAG: при `p>0.05` внутри страт конфаундер «объяснён» degree, при `p≪0.05` — остаётся независимый эффект (ошибка спецификации DAG).

Дополнительно считаем частичную корреляцию `corr(Exchange_Hot, y | degree)` через остатки линейной регрессии.



In [ ]:
# стратификация по degree bins (квантили)
# degree bins by quantiles (robust to duplicates)
bins = [-1, 4, 10, 500]  # straddle thr=4 so all strata have hubs & non-hubs  # ensures hubs (deg>=4) appear in 2 strata
deg_q = pd.cut(df["_deg"], bins=bins, labels=["low","mid","high"], include_lowest=True)
ct_results = []
for grp in deg_q.cat.categories:
    mask = (deg_q == grp)
    sub = df.loc[mask]
    # 2×2 таблица: rows Exchange_Hot, cols y
    if sub["Exchange_Hot"].nunique() < 2:
        # нет вариации конфаундера в страте — тест не определён
        ct_results.append((str(grp), int(mask.sum()), float("nan"), float("nan"), float("nan")))
        print(f"stratum {grp}: n={int(mask.sum()):,}  — no variation in Exchange_Hot (all {sub['Exchange_Hot'].iloc[0]}) → skip χ²")
        continue
    tbl = pd.crosstab(sub["Exchange_Hot"], sub["y"])
    for v in [0,1]:
        if v not in tbl.index: tbl.loc[v]=0
        if v not in tbl.columns: tbl[v]=0
    tbl = tbl.sort_index().sort_index(axis=1).values
    if tbl.min() == 0:
        tbl = tbl + 0.5  # Haldane correction
    chi2, p, dof, exp = chi2_contingency(tbl, correction=False)
    a,b,c,d = tbl[1,1], tbl[1,0], tbl[0,1], tbl[0,0]
    or_val = (a*d)/(b*c) if b*c>0 else np.nan
    ct_results.append((str(grp), int(mask.sum()), float(or_val), float(chi2), float(p)))
    print(f"stratum {grp}: n={int(mask.sum()):,}  OR={or_val:.3f}  χ²={chi2:.2f}  p={p:.3g}  table={tbl.tolist()}")

ct_df = pd.DataFrame(ct_results, columns=["degree_bin","n","OR","chi2","p"])
display(ct_df)

# частичная корреляция Exchange_Hot — y | degree (через остатки)
from scipy.stats import pearsonr
# линейная проекция на degree
Xc = df["_deg"].values.astype(float)
y_arr = df["y"].values.astype(float)
eh_arr = df["Exchange_Hot"].values.astype(float)
# де-mean и регрессия OLS 1D: slope = cov/var
def residuals(a, x):
    slope = np.cov(a, x, bias=True)[0,1] / (np.var(x)+1e-12)
    intercept = a.mean() - slope*x.mean()
    return a - (slope*x + intercept)
res_y = residuals(y_arr, Xc)
res_eh = residuals(eh_arr, Xc)
r_partial, p_partial = pearsonr(res_eh, res_y)
print(f"\nPartial corr(Exchange_Hot, y | degree): r={r_partial:.4f}  p={p_partial:.3g}")
print(f"Marginal corr(Exchange_Hot, y): r={np.corrcoef(eh_arr, y_arr)[0,1]:.4f}")







In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
# p-value по стратам
sns.barplot(data=ct_df, x="degree_bin", y="p", hue="degree_bin", palette="viridis", ax=axes[0], legend=False)
axes[0].axhline(0.05, color="red", linestyle="--", label="α=0.05")
axes[0].set_title("Falsification: p-value χ² внутри degree-страт")
axes[0].set_ylabel("p (χ² Exchange_Hot vs y)")
axes[0].legend()
for i, row in enumerate(ct_df.itertuples()):
    axes[0].text(i, row.p+0.02, f"p={row.p:.2g}\nOR={row.OR:.2f}", ha="center", fontsize=7)

# OR по стратам
sns.barplot(data=ct_df, x="degree_bin", y="OR", hue="degree_bin", palette="magma", ax=axes[1], legend=False)
axes[1].axhline(1.0, color="grey", linestyle=":")
axes[1].set_title("Odds ratio illicit | Exchange_Hot внутри страт")
axes[1].set_ylabel("OR")
for i, row in enumerate(ct_df.itertuples()):
    axes[1].text(i, row.OR+0.05, f"{row.OR:.2f}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()
print("Если p≪0.05 во всех стратах — Exchange_Hot несёт сигнал сверх degree (DAG недо-специфицирован). "
      "Если p>0.05 — связь объясняется degree, фальсификация не отвергает упрощённый DAG. "
      f"Наблюдаем: partial r={r_partial:.3f} (vs marginal {np.corrcoef(eh_arr, y_arr)[0,1]:.3f}) — эффект после контроля degree.")



## 6. Выводы

- **Distance ≠ причинность.** Retrieval по Euclidean (k=5) находит корреляционные соседи: illicit ближе к пулу (mean 6 vs 10–13 для licit), но `Exchange_Hot` (1.7% illicit vs 10.9% baseline) создаёт spurious граф-близость: хаб топологически рядом со многими illicit, оставаясь licit. Гистограмма hub vs non-hub это демонстрирует.
- **DAG-фильтр отсекает spurious.** Правило `passed = ¬(confounded & distance_explained)` с `ε=5.0` сняло ~12–13% top-5 пар (`pass_rate≈0.87`), переводя query на следующий ближайший anchor. PR-AUC на full test: ~0.53 → ~0.52–0.53 — фильтр платит небольшим recall за интерпретируемость; цель — не максимизировать AP, а убрать граф-конфаундер.
- **Sensitivity — честная оценка неуверенности.** E-value (`RR+√RR(RR-1)`, Tier1 `E>2.0`) и Rosenbaum `Γ*=1+effect` показывают, какой силы должен быть ненаблюдаемый `Owner/Intent`, чтобы инвертировать вывод. На данных лишь ~5–10% алертов Tier1 — большинство чувствительны к скрытому смешиванию.
- **Falsification не отвергает прокси-DAG, но и не подтверждает его.** `Exchange_Hot ⟂ y | degree` даёт `p` и `OR` внутри страт + `partial r≈` — если `p` мал, хаб несёт сигнал сверх degree (спецификация упрощена).
- **Ограничения и ponytail.** Это лишь наблюдаемый прокси-DAG. Полноценный DAG — PC-algorithm / NOTEARS на (feat, degree, PageRank, time) + скрытые переменные (Owner, Intent) через sensitivity bounds и negative controls. Следующий шаг Spillety — графовый скор с явным моделированием `Exchange_Hot` как confounder + калибровка на valid с учётом `Γ*`.


